# Reconciliation Checks

Compares business keys between medallion layers to prove that nothing is lost or
unexpectedly gained in transit.

**Bronze → silver:** the only acceptable gap is records sent to quarantine.
**Reference vs stream:** the registry updates more slowly than the live feed, so
objects can appear in one and not the other in both directions.

Results are appended to `reconciliation_results`, which gives a metric history
over time and is what the data-quality alert queries.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

dbutils.widgets.text("catalog","")
dbutils.widgets.text("schema","")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

results_table = f"{catalog}.{schema}.reconciliation_results"
check_ts = datetime.now()

In [0]:
def reconcile(name, source_df, source_key, source_name, target_df, target_key, target_name):
    # Cast to string - the same business key can be INT in one layer and STRING in
    # another (route_id in GTFS vs routeId in the stream).
    source_keys = source_df.select(F.col(source_key).cast("string").alias("k")).distinct()
    target_keys = target_df.select(F.col(target_key).cast("string").alias("k")).distinct()

    # exceptAll is set subtraction - keys present on one side but absent from the other.
    # Both directions matter. One means data loss, the other unexpected arrivals.
    missing_in_target = source_keys.exceptAll(target_keys)
    missing_in_source = target_keys.exceptAll(source_keys)

    source_n, target_n = source_keys.count(), target_keys.count()
    missing_target_n = missing_in_target.count()
    missing_source_n = missing_in_source.count()

    print(f"\n=== {name} ===")
    print(f"  source keys: {source_n:,}")
    print(f"  target keys: {target_n:,}")
    print(f"  in source but not target: {missing_target_n:,}")
    print(f"  in target but not source: {missing_source_n:,}")

    # Persist so the metric can be tracked over time and queried by an alert
    (spark.createDataFrame(
        [(name, source_name, target_name,
          source_n, target_n, missing_target_n, missing_source_n)],
        "check_name string, source_table string, target_table string, "
        "source_keys long, target_keys long, missing_in_target long, missing_in_source long")
     .withColumn("checked_at", F.lit(check_ts).cast("timestamp"))
     .write.format("delta").mode("append").saveAsTable(results_table))

    return missing_in_target, missing_in_source

In [0]:
# Bronze -> silver: nothing should be lost except records sent to quarantine
_ = reconcile("vehicles: bronze -> silver",
    spark.read.table(f"{catalog}.{schema}.bronze_vehicles"), "vehicleCode", "bronze_vehicles",
    spark.read.table(f"{catalog}.{schema}.silver_vehicles_scd").filter("is_current = true"),
    "vehicleCode", "silver_vehicles_scd")

_ = reconcile("routes: bronze -> silver",
    spark.read.table(f"{catalog}.{schema}.bronze_gtfs_routes"), "route_id", "bronze_gtfs_routes",
    spark.read.table(f"{catalog}.{schema}.silver_routes_scd").filter("is_current = true"),
    "route_id", "silver_routes_scd")

# Reference batch vs stream, registry updates more slowly than the live feed
# so some objects are seen moving before they are published and vice versa.
veh_missing, veh_extra = reconcile("vehicles: reference vs stream",
    spark.read.table(f"{catalog}.{schema}.silver_vehicles_scd").filter("is_current = true"),
    "vehicleCode", "silver_vehicles_scd",
    spark.read.table(f"{catalog}.{schema}.dim_vehicle"), "vehicleCode", "dim_vehicle")

rt_missing, rt_extra = reconcile("routes: reference vs stream",
    spark.read.table(f"{catalog}.{schema}.silver_routes_scd").filter("is_current = true"),
    "route_id", "silver_routes_scd",
    spark.read.table(f"{catalog}.{schema}.dim_route"), "routeId", "dim_route")

In [0]:
# The stream vs registry gap mixes two distinct problems, so report them apart
# D-prefixed codes come from the schema-evolution demo producer (route 999,
# headsign DEMO). They are test artefacts.
demo_n = veh_extra.filter(F.col("k").startswith("D")).count()
real_n = veh_extra.filter(~F.col("k").startswith("D")).count()
print(f"Test artefacts leaked into the stream (D-prefixed): {demo_n}")
print(f"Real vehicles not yet published in the registry:    {real_n}")

print("\nVehicles seen in the stream but absent from the registry:")
# display(veh_extra.orderBy("k"))

print("\nRoutes seen in the stream but absent from GTFS:")
# display(rt_extra.orderBy("k"))

In [0]:
# Metric history: each run appends one row per check, which is what the alert queries

# display(spark.read.table(results_table).orderBy(F.col("checked_at").desc()))